# Giga Meter — Clean (data preparation)

Pull one country's measurements, clean them with a **visible filter funnel**, and
export the analysis-ready dataset the downstream notebooks consume.

The pipeline itself lives in **`helpers/prepare.py`** (shared with
`meter_explorer_02` — one source of truth, no duplicated cleaning logic). This
notebook keeps the two decisions that deserve eyes on them before anything is
dropped:

1. **Latency outlier cutoff** — inspect the distribution, then choose.
2. **School-hours window** — inspect the time-of-day profile, then choose.

Non-interactive form: `python scripts/download_data.py <ISO3> [--admin1 …]`.

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, str(Path.cwd() / "helpers"))
from prepare import (load_country, latency_distribution, latency_cutoff_candidates,
                     prepare_country, export_clean)
from eda_helpers import resolve_country
try:
    import giga_chart_style   # Giga fonts/palette applied on import
except Exception:
    pass
print("✓ Imports complete (pipeline: helpers/prepare.py)")

In [ ]:
# =============================================================================
# COUNTRY & CONFIG — set ONE code; iso2 / name / timezone resolve automatically
# =============================================================================
COUNTRY = "ALB"   # ISO3 code or country name (registry: helpers/country_reference.json)

_c = resolve_country(COUNTRY)
COUNTRY_ISO3, COUNTRY_ISO2, COUNTRY_NAME, TIMEZONE = _c["iso3"], _c["iso2"], _c["name"], _c["timezone"]

# ── Data loading ─────────────────────────────────────────────────────────────
USE_CACHED_DATA = True        # True: parquet caches; False: pull/refresh from Trino
MEASUREMENT_SOURCE = None     # rt_source filter; None = all sources
ROWLEVEL_WINDOW_DAYS = None   # e.g. 365 -> only load the trailing year (big countries)
LOAD_COLUMNS = None           # e.g. a column list -> prune columns at read

# ── Cleaning scope ───────────────────────────────────────────────────────────
ADMIN1_FILTER = None          # e.g. "Eastern Cape" scopes everything to one region
SERVER_FILTER = True          # keep dominant measurement server(s) only
SERVER_PERC_THRESHOLD = 0.30  # a server is "dominant" with >= this share of tests

# ── Scope defaults carried into the params json for downstream notebooks ─────
EDUCATION_LEVEL = 'Secondary' # e.g. 'Secondary', 'Primary', or None for all levels
YEARS = None                  # None -> auto-detect from the data; or ['2025','2026']
MIN_DAYS_MONTH = 10           # a school-month is "analysable" with >= this many measured days
MIN_WEEKDAYS_MEASURED = 10    # min weekdays with data for detailed per-school analysis
THR, THR_UL, THR_LAT = 20, 10, 100   # download / upload (Mbps), latency (ms) working thresholds

print(f"✓ {COUNTRY_NAME} ({COUNTRY_ISO3}/{COUNTRY_ISO2}) · timezone {TIMEZONE}"
      + (f"  [country spans {len(_c['timezones'])} zones]" if len(_c['timezones']) > 1 else ""))

In [ ]:
# =============================================================================
# LOAD — master + measurements + registration (raw; nothing dropped yet)
# =============================================================================
L = load_country(COUNTRY,
                 use_cached=USE_CACHED_DATA,
                 measurement_source=MEASUREMENT_SOURCE,
                 rowlevel_window_days=ROWLEVEL_WINDOW_DAYS,
                 load_columns=LOAD_COLUMNS)
m, master, r = L.m, L.master, L.registration

## Latency outlier cutoff — inspect, then choose

The cutoff is a **reviewed decision**: look at this country's distribution and the
candidates, then set `LATENCY_CUTOFF` below (a number in ms, or
`'p95' | 'p99' | 'p99.5' | 'iqr' | 'modz'`).

In [ ]:
_cand = latency_distribution(L.m, country_name=f"— {COUNTRY_NAME}")

LATENCY_CUTOFF = "p99"   # <- override after reviewing the histogram/candidates

## School-hours window — inspect, then choose

`measurement_time_window` (school_hours / off_hours) drives per-school IQB and every
"school hours" analysis downstream. Review the profile; override if this country's
school day differs.

In [ ]:
_hr = pd.to_datetime(L.m["timestamplocal"]).dt.hour

SCHOOL_HOURS_START, SCHOOL_HOURS_END = 7, 16   # <- override, then run on (inclusive bounds)

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.hist(_hr, bins=range(25), alpha=0.85, edgecolor="white")
ax.axvspan(SCHOOL_HOURS_START, SCHOOL_HOURS_END + 1, alpha=0.15)
ax.set_xticks(range(0, 25, 2)); ax.set_xlabel("local hour"); ax.set_ylabel("measurements")
ax.set_title(f"Time-of-day profile — {COUNTRY_NAME} (shaded = school-hours window)")
plt.tight_layout(); plt.show()

## Prepare — clean with a visible funnel

Every dropped row is counted; impossible values are nulled (rows kept) and
reported separately. `m` = analysis frame · `m_original` = clean but unfiltered
(the base for drop-off / time-series work).

In [ ]:
P = prepare_country(L, latency_cutoff=LATENCY_CUTOFF,
                    server_filter=SERVER_FILTER, server_pct=SERVER_PERC_THRESHOLD,
                    admin1=ADMIN1_FILTER,
                    school_hours=(SCHOOL_HOURS_START, SCHOOL_HOURS_END))

m, m_original = P.m, P.m_original
LATENCY_OUTLIER_THRESHOLD = P.latency_threshold_ms
print(m["measurement_time_window"].value_counts().to_string())

## Export the clean dataset

In [ ]:
export_clean(P, extra_params={"scope_defaults": {
    "education_level": EDUCATION_LEVEL, "years": YEARS,
    "min_days_month": MIN_DAYS_MONTH,
    "thr_download_mbps": THR, "thr_upload_mbps": THR_UL, "thr_latency_ms": THR_LAT,
    "min_weekdays_measured": MIN_WEEKDAYS_MEASURED}})